In [1]:
# Cell 1 — Imports
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
import warnings; warnings.filterwarnings('ignore')
from src.data import load_train, load_test, CLASSES
from src.features import build_features
from src.metrics import compute_map, print_results
from src.submission import save_submission

print('All imports OK')
print(f'Classes: {CLASSES}')

All imports OK
Classes: ['Clutter', 'Cormorants', 'Pigeons', 'Ducks', 'Geese', 'Gulls', 'Birds of Prey', 'Waders', 'Songbirds']


In [2]:
# Cell 2 — Load & inspect data
train = load_train()
test  = load_test()

print(f'Train: {train.shape}, Test: {test.shape}')
print('\nClass distribution:')
print(train.bird_group.value_counts())

Train: (2601, 16), Test: (1872, 9)

Class distribution:
bird_group
Gulls            1503
Songbirds         483
Pigeons           122
Waders            120
Birds of Prey     108
Clutter            84
Geese              83
Ducks              58
Cormorants         40
Name: count, dtype: int64


In [3]:
# Cell 3 — Feature extraction
print('Extracting train features...')
X_df = build_features(train)        # all 3 feature sets, inf/nan handled
print('Extracting test features...')
X_test_df = build_features(test)
print(f'Shape: train={X_df.shape}, test={X_test_df.shape}')

Extracting train features...
Extracting test features...
Shape: train=(2601, 73), test=(1872, 73)


In [4]:
# Cell 4 — Encode target & class weights
le = LabelEncoder().fit(CLASSES)
y  = le.transform(train['bird_group'])
n_classes = len(CLASSES)
X      = X_df.values
X_test = X_test_df.values
feature_names = list(X_df.columns)

# Inverse-frequency class weights (used by XGB/CB)
class_counts = np.bincount(y)
class_weights = (1 / class_counts) / (1 / class_counts).sum() * n_classes

print(f'Classes: {le.classes_}')
print(f'y distribution: {np.bincount(y)}')
print(f'Class weights: {class_weights.round(3)}')
print(f'Feature count: {len(feature_names)}')

Classes: ['Birds of Prey' 'Clutter' 'Cormorants' 'Ducks' 'Geese' 'Gulls' 'Pigeons'
 'Songbirds' 'Waders']
y distribution: [ 108   84   40   58   83 1503  122  483  120]
Class weights: [0.88  1.131 2.375 1.638 1.145 0.063 0.779 0.197 0.792]
Feature count: 73


In [5]:
# Cell 5 — 5-fold CV with SMOTE + three-model training
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros((len(X), n_classes))
oof_xgb = np.zeros((len(X), n_classes))
oof_cb  = np.zeros((len(X), n_classes))
test_lgb = np.zeros((len(X_test), n_classes))
test_xgb = np.zeros((len(X_test), n_classes))
test_cb  = np.zeros((len(X_test), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # SMOTE: oversample all minority classes to match majority inside fold
    # NOTE: applied only to training split — validation remains unaugmented original data
    smote = SMOTE(sampling_strategy='not majority', random_state=42 + fold)
    X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)
    print(f'Fold {fold}: {len(y_tr)} → {len(y_tr_sm)} samples after SMOTE')

    # --- LightGBM ---
    lgb_params = dict(
        objective='multiclass', num_class=n_classes, metric='multi_logloss',
        learning_rate=0.05, num_leaves=47, max_depth=7,
        min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, is_unbalance=True,
        verbose=-1, seed=42, n_jobs=-1
    )
    dtrain = lgb.Dataset(X_tr_sm, label=y_tr_sm, feature_name=feature_names)
    dval   = lgb.Dataset(X_va, label=y_va, reference=dtrain)
    lgb_model = lgb.train(lgb_params, dtrain, 2000, valid_sets=[dval],
                           callbacks=[lgb.early_stopping(80), lgb.log_evaluation(200)])
    oof_lgb[va_idx]  = lgb_model.predict(X_va)
    test_lgb        += lgb_model.predict(X_test) / N_FOLDS

    # --- XGBoost ---
    sample_w = class_weights[y_tr_sm]
    xgb_model = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', early_stopping_rounds=80,
        seed=42, n_jobs=-1, verbosity=0
    )
    xgb_model.fit(X_tr_sm, y_tr_sm, sample_weight=sample_w,
                  eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx]  = xgb_model.predict_proba(X_va)
    test_xgb        += xgb_model.predict_proba(X_test) / N_FOLDS

    # --- CatBoost ---
    cb_model = CatBoostClassifier(
        iterations=2000, learning_rate=0.05, depth=6,
        loss_function='MultiClass', eval_metric='TotalF1',
        class_weights=class_weights.tolist(),
        early_stopping_rounds=80, random_seed=42,
        verbose=200
    )
    cb_model.fit(X_tr_sm, y_tr_sm, eval_set=(X_va, y_va))
    oof_cb[va_idx]  = cb_model.predict_proba(X_va)
    test_cb        += cb_model.predict_proba(X_test) / N_FOLDS

print('\nTraining complete!')

Fold 0: 2080 → 10818 samples after SMOTE
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[109]	valid_0's multi_logloss: 0.602177
0:	learn: 0.6616305	test: 0.3912357	best: 0.3912357 (0)	total: 110ms	remaining: 3m 40s
200:	learn: 0.9766442	test: 0.6399665	best: 0.6399665 (199)	total: 4.36s	remaining: 39s
400:	learn: 0.9877384	test: 0.6842719	best: 0.6889031 (326)	total: 8.38s	remaining: 33.4s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.6889030558
bestIteration = 326

Shrink model to first 327 iterations.
Fold 1: 2081 → 10818 samples after SMOTE
Training until validation scores don't improve for 80 rounds
Early stopping, best iteration is:
[104]	valid_0's multi_logloss: 0.598574
0:	learn: 0.6447419	test: 0.3761682	best: 0.3761682 (0)	total: 45.6ms	remaining: 1m 31s
200:	learn: 0.9733765	test: 0.5784416	best: 0.5844851 (197)	total: 4.31s	remaining: 38.6s
400:	learn: 0.9870698	test: 0.6295484	best: 0.6303587 (379)	total

In [6]:
# Cell 6 — Optimize ensemble weights on OOF
def blended(weights):
    w = np.abs(weights); w /= w.sum()
    pred = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cb
    mAP, _ = compute_map(y, pred)
    return -mAP   # minimize → maximize mAP

res = minimize(blended, [1/3, 1/3, 1/3], method='Nelder-Mead',
               options={'maxiter': 1000, 'xatol': 1e-5})
w_opt = np.abs(res.x); w_opt /= w_opt.sum()
print(f'Optimal weights — LGB: {w_opt[0]:.3f}, XGB: {w_opt[1]:.3f}, CB: {w_opt[2]:.3f}')

Optimal weights — LGB: 0.489, XGB: 0.412, CB: 0.099


In [7]:
# Cell 7 — Evaluate CV mAP
oof_blend = w_opt[0]*oof_lgb + w_opt[1]*oof_xgb + w_opt[2]*oof_cb
cv_mAP, per_class = compute_map(y, oof_blend)
print_results(cv_mAP, per_class, label='adda_smote_ensemble')


  adda_smote_ensemble

  Overall mAP: 0.7155

  Clutter        : 0.5710 <-- weak
  Cormorants     : 0.9376
  Pigeons        : 0.2145 <-- weak
  Ducks          : 0.6454
  Geese          : 0.7857
  Gulls          : 0.9553
  Birds of Prey  : 0.8865
  Waders         : 0.8274
  Songbirds      : 0.6160


In [8]:
# Cell 8 — Save submission
test_blend = w_opt[0]*test_lgb + w_opt[1]*test_xgb + w_opt[2]*test_cb
save_submission(test_blend, name='adda_smote_ensemble', cv_map=cv_mAP)
print('Saved. Also check submissions/ folder for versioned file.')

Saved: adda_smote_ensemble_0.7155_20260213_1709.csv (1872 rows)
Saved. Also check submissions/ folder for versioned file.
